# 🚀 Agent 65: DPO Fine-Tuning Llama 3.1 8B on Free Google Colab

This notebook fine-tunes **Meta-Llama-3.1-8B-Instruct** using **Direct Preference Optimization (DPO)** and **Unsloth** on a **Free Google Colab T4 GPU**.

- 🧠 **Model**: `Meta-Llama-3.1-8B-Instruct` (8 Billion parameters, 128k context, Gemini/ChatGPT-level reasoning)
- 🎯 **Method**: Direct Preference Optimization (DPO) with `(chosen, rejected)` contrastive learning
- ⏱️ **Training Time**: ~18–22 minutes on free T4 GPU
- 💾 **VRAM Usage**: ~8.5 GB (comfortably fits on the free 16 GB T4)
- 📦 **Output**: Merged `agent65-8b-dpo-q4_k_m.gguf` (~4.8 GB) ready for local **Ollama** deployment with **zero API keys**!

### Step 1: Install High-Performance Unsloth Stack
Run this cell to install Unsloth and DPO training libraries.

In [ ]:
!pip install --upgrade pip
!pip install unsloth unsloth_zoo
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl peft accelerate bitsandbytes datasets triton
print("✅ Unsloth & DPO dependencies installed successfully!")

### Step 2: Load Base Llama 3.1 8B in 4-bit Quantization
Loads `Meta-Llama-3.1-8B-Instruct` pre-quantized in 4-bit and attaches QLoRA adapters.

In [ ]:
import torch
from unsloth import FastLanguageModel, PatchDPOTrainer

# Patch DPO Trainer for memory optimization and 2x faster execution
PatchDPOTrainer()

max_seq_length = 2048
dtype = None # Auto detection (Float16 or Bfloat16)
load_in_4bit = True

print("[*] Loading Llama 3.1 8B Instruct (4-bit)... please wait ~2 mins...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Attach LoRA Adapters for parameter-efficient fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✅ Llama 3.1 8B model loaded with LoRA adapters!")

### Step 3: Upload `dpo_train_dataset.jsonl`
Click the **Choose Files** button to upload `dpo_train_dataset.jsonl` from your `StudentHelpdesk/backend/data/` folder.

In [ ]:
from google.colab import files
import os
import glob
from datasets import load_dataset

print("[*] Please upload `dpo_train_dataset.jsonl` from your local backend/data folder:")
uploaded = files.upload()

# Auto-detect uploaded JSONL file
jsonl_files = [f for f in glob.glob("*.jsonl") if "val" not in f]
target_file = jsonl_files[0] if jsonl_files else "dpo_train_dataset.jsonl"
print(f"[*] Ingesting DPO contrastive dataset: {target_file}")

dataset = load_dataset("json", data_files=target_file, split="train")
print(f"✅ Successfully loaded {len(dataset)} DPO contrastive pairs (prompt, chosen, rejected)!")

### Step 4: Run DPO Training (~18–20 minutes on T4 GPU)
DPO mathematically teaches the model to favor empathetic, factually grounded answers and penalize failure modes (fee dumps for headaches, raw LaTeX bugs).

In [ ]:
from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    warmup_ratio = 0.1,
    max_steps = 60, # Optimal sweet spot for 8B DPO (~1-2 epochs, prevents overfitting)
    learning_rate = 5e-6, # Low learning rate ensures stable alignment
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    lr_scheduler_type = "cosine",
    seed = 3407,
    output_dir = "dpo_outputs",
    beta = 0.1, # DPO temperature controlling reference divergence
    max_length = 2048,
    max_prompt_length = 1024,
)

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,
    args = dpo_config,
    train_dataset = dataset,
    tokenizer = tokenizer,
)

print("🚀 Starting DPO alignment on Llama 3.1 8B...")
trainer_stats = dpo_trainer.train()
print("🎉 DPO Alignment Complete! Model has learned to favor empathy and reject failure modes.")

### Step 5: Export to GGUF (Q4_K_M) & Download
Merges LoRA adapter weights and exports directly to a 4-bit quantized GGUF format ready for local Ollama execution.

In [ ]:
# Export to GGUF with Q4_K_M quantization (~4.8 GB)
print("[*] Merging LoRA weights and quantizing to Q4_K_M GGUF format (~4-5 mins)...")
model.save_pretrained_gguf("agent65_8b_dpo_model", tokenizer, quantization_method = "q4_k_m")

# Locate generated GGUF file
import glob
from google.colab import files

gguf_files = glob.glob("agent65_8b_dpo_model*/**/*.gguf", recursive=True) + glob.glob("*8b*.gguf") + glob.glob("*.gguf")
print(f"[*] Found GGUF models: {gguf_files}")

if gguf_files:
    target_gguf = gguf_files[0]
    print(f"[*] Triggering browser download for: {target_gguf}...")
    files.download(target_gguf)
else:
    print("[!] Check the left sidebar Files tab under `agent65_8b_dpo_model/` to download your .gguf file manually.")

### Step 6: Deploy Locally in Ollama (On your Windows PC)

Once the download completes on your computer:
1. Move the downloaded `.gguf` file to your project directory:
   `c:\StudentHelpdesk\backend\models\agent65-8b-dpo-q4_k_m.gguf`

2. Register the model in Ollama:
   ```powershell
   ollama create agent65-8b -f backend/models/Modelfile.agent65-8b
   ```

3. Your Agent 65 backend will **automatically detect `agent65-8b`** as the primary cognitive engine running 100% locally with zero external API keys!